In [1]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import json

In [2]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file
openai_keys = os.getenv("OPENAI_API_KEY")
if not openai_keys:
    raise ValueError("Please provide an OpenAI API key.")

In [3]:
## Model
model = ChatOpenAI(model="gpt-4o-mini")

In [11]:
import json
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Path to your JSONL file
jsonl_file_path = "/Users/danielstephens/Desktop/Annotune-v2/synthetic-first-contact-plot-summaries-20241023-130150.jsonl"

def load_and_chunk_jsonl(jsonl_file_path):
    # Initialize the text splitter with desired chunk size
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=7000,  # Adjust chunk size based on your needs
        chunk_overlap=200  # Optional overlap between chunks
    )
    
    # Extract and chunk each document from the JSONL file
    chunks = []
    with open(jsonl_file_path, 'r') as file:
        for line in file:
            document = json.loads(line.strip())  # Load each JSON object
            
            # Extract fields
            summary = document.get("summary", "")
            setting = document.get("setting", "")
            style = document.get("style", "")
            themes = [document.get("theme_1", ""), document.get("theme_2", "")]
            
            if summary:
                # Split the summary text into chunks
                summary_chunks = text_splitter.split_text(summary)
                
                # Create structured chunks with additional metadata
                for chunk in summary_chunks:
                    structured_chunk = {

                        "setting": setting,
                        "style": style,
                        "themes": themes,
                        "content": chunk
                    }
                    chunks.append(structured_chunk)
    
    return chunks

# Example usage
chunks = load_and_chunk_jsonl(jsonl_file_path)
for i, chunk in enumerate(chunks[:5]):  # Print the first few structured chunks as a sample
    print(f"Chunk {i+1}:")
    print(f"Setting: {chunk['setting']}")
    print(f"Style: {chunk['style']}")
    print(f"Themes: {chunk['themes']}")
    print(f"Content:\n{chunk['content']}\n")



Chunk 1:
Setting: Research facilities or laboratories: Controlled environments where scientists and experts can study and interact with the alien intelligence.
Style: Biopunk: This style combines science fiction with biotechnology, genetics, and biologically inspired innovations, often with a focus on the ethics of scientific discovery. Examples: Paolo Bacigalupi, Margaret Atwood, and China Miéville.
Themes: ["Humanity's place in the universe: Questioning humanity's significance, morality, and purpose in the face of a non-human intelligence.", 'Communication and understanding: Investigating the challenges and possibilities of communication between humans and non-human intelligences.']
Content:
In the labyrinthine corridors of the Nyxion Research Facility, nestled within the sprawling metropolis of New Tethys, a team of scientists led by the enigmatic Dr. Lyra Flynn embarked on a groundbreaking experiment. Their aim was to establish a rapport with an extraterrestrial entity, christened 

In [ ]:
from langchain.vectorstores import Chroma
# Initialize embeddings and vector database
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_db = Chroma(collection_name="synthetic", persist_directory="database/chroma", embedding_function=embeddings)
for chunk in chunks:
    vector_db.add_texts([json.dumps(chunk)])


In [15]:
# Making a retriever 

retriever  = vector_db.as_retriever(search_kwargs={"k": 30})

In [ ]:
retriever.invoke("What is the best spaceship?")

In [20]:
import json

# Path to your JSONL file
jsonl_file_path = "/Users/danielstephens/Desktop/Annotune-v2/synthetic-first-contact-plot-summaries-20241023-130150.jsonl"

def get_unique_attributes(jsonl_file_path):
    # Sets to hold unique values
    unique_themes = set()
    unique_settings = set()
    unique_styles = set()
    
    # Read each JSON object line by line
    with open(jsonl_file_path, 'r') as file:
        for line in file:
            document = json.loads(line.strip())  # Load each JSON object
            
            # Add themes, setting, and style to respective sets
            theme_1 = document.get("theme_1", "")
            theme_2 = document.get("theme_2", "")
            setting = document.get("setting", "")
            style = document.get("style", "")
            
            if theme_1:
                unique_themes.add(theme_1)
            if theme_2:
                unique_themes.add(theme_2)
            if setting:
                unique_settings.add(setting)
            if style:
                unique_styles.add(style)
    
    # Convert sets to sorted lists for easier reading
    return {
        "themes": sorted(unique_themes),
        "settings": sorted(unique_settings),
        "styles": sorted(unique_styles)
    }

# Extract unique themes, settings, and styles
unique_attributes = get_unique_attributes(jsonl_file_path)
print("Unique Themes:", unique_attributes["themes"])
print("Unique Settings:", unique_attributes["settings"])
print("Unique Styles:", unique_attributes["styles"])


Unique Themes: ['Communication and understanding: Investigating the challenges and possibilities of communication between humans and non-human intelligences.', "Cultural and societal implications: Examining how humanity's institutions, values, and norms might be affected by contact with an alien intelligence.", 'Ethics and morality: Delving into the moral and ethical dilemmas that arise from encountering a non-human intelligence, such as the potential for exploitation or conflict.', "Humanity's place in the universe: Questioning humanity's significance, morality, and purpose in the face of a non-human intelligence.", 'The Other: Exploring the nature of the alien intelligence, its motivations, and its place in the universe.', "The impact on human identity: Investigating how contact with a non-human intelligence might challenge or change humanity's self-perception and sense of identity.", 'The unknown and the unknowable: Exploring the limits of human knowledge and understanding in the fa

In [26]:
PROMPT_TEMPLATE = """
                    Context: I have a collection of documents stored in a vector database. Each document is tagged with specific themes, settings,
                    and styles, which provide insight into its focus, location, and narrative approach. Based on the following attributes, 
                    generate a set of general questions that reflect meaningful inquiries across the data. Ensure each question is engaging, 
                    relevant to the data context, and prompts thoughtful exploration.
                    Extracted Context : {context}
                    
                    Themes: ['Communication and understanding: Investigating the challenges and possibilities of communication between humans 
                                and non-human intelligences.', "Cultural and societal implications: Examining how humanity's institutions, 
                                values, and norms might be affected by contact with an alien intelligence.", 'Ethics and morality: Delving 
                                into the moral and ethical dilemmas that arise from encountering a non-human intelligence, such as the potential 
                                for exploitation or conflict.', "Humanity's place in the universe: Questioning humanity's significance, morality, 
                                and purpose in the face of a non-human intelligence.", 'The Other: Exploring the nature of the alien intelligence, 
                                its motivations, and its place in the universe.', "The impact on human identity: Investigating how contact with a 
                                non-human intelligence might challenge or change humanity's self-perception and sense of identity.", 
                                'The unknown and the unknowable: Exploring the limits of human knowledge and understanding in the face of an alien 
                                intelligence that may operate under fundamentally different principles.']
                    
                    Settings: ['Asteroids, moons, or planets: Uninhabited or partially explored celestial bodies that can serve as a neutral ground 
                                for first contact.', 'Parallel universes or alternate dimensions: Settings that allow for the exploration of different realities 
                                and the implications of first contact across multiple realities.', 'Post-apocalyptic or dystopian futures: Worlds 
                                that have been ravaged by war, environmental disaster, or other catastrophes, which can serve as a backdrop for a 
                                desperate or opportunistic first contact.', 'Remote or isolated locations: Deserts, jungles, or other areas that can 
                                provide a sense of separation from the rest of humanity.', 'Research facilities or laboratories: Controlled environments 
                                where scientists and experts can study and interact with the alien intelligence.', 
                                'Simulated environments: Virtual reality or artificial environments that can blur the lines between reality and fantasy.',
                                'Space stations or colonies: Isolated and vulnerable, these settings can heighten the sense of tension and uncertainty.', 
                                'Spaceships: A self-contained environment that can facilitate a more intimate, claustro- phobic encounter with the alien 
                                intelligence.', 'Urban centers: Cities or metropolitan areas that can highlight the contrast between hu- man civilization
                                and the alien presence.']
                    
                    Styles: ['Alternate History: This style explores the consequences of historical events turning out differently, often featuring 
                                parallel universes or divergent timelines. Examples: Philip K. Dick, Harry Turtledove, and S.M. Stirling.', 
                                'Biopunk: This style combines science fiction with biotechnology, genetics, and biologically inspired innovations, often with a 
                                focus on the ethics of scientific discovery. Examples: Paolo Bacigalupi, Margaret Atwood, and China Miéville.', 
                                'Cyberpunk: This style combines high-tech advancements with a focus on virtual reality, artificial intelligence, and the impact 
                                of technology on society. Examples: William Gibson, Bruce Sterling, and Neal Stephenson.', 
                                'Hard Science Fiction: This style focuses on scientific accuracy and technical details, often featuring engineers, scientists, 
                                and inventors as main characters. Examples: Isaac Asimov, Arthur C. Clarke, and Kim Stanley Robinson.', 
                                'Military Science Fiction: This style emphasizes military conflict, strategy, and technology in a science fiction setting, often 
                                featuring soldiers, pilots, or mercenaries as main characters. Examples: Joe Haldeman, Lois McMaster Bujold, and 
                                John Scalzi.', 'New Wave: This style emerged in the 1960s, characterized by experimental narrative structures, 
                                psychological introspection, and a focus on the human condition. Examples: J.G. Ballard, Thomas M. Disch, and 
                                Ursula K. Le Guin.', 'Slipstream: This style blends science fiction with surrealism, often featuring dreamlike
                                  narratives and unconventional storytelling. Examples: Thomas Pynchon, Don DeLillo, and Rudy Rucker.', 
                                  'Soft Science Fiction: This style emphasizes social and psychological aspects, often exploring the human 
                                  condition, emotions, and relationships in a science fiction setting. 
                                  Examples: Ray Bradbury, Ursula K. Le Guin, and Octavia Butler.', 
                                  'Space Opera: This style typically involves epic, sprawling narratives that explore the galaxy, 
                                  alien civilizations, and interstellar conflicts. Examples: Frank Herbert, E.E. Smith, and Cixin Liu.', 
                                  'Time Travel: This style involves stories that manipulate time, exploring the consequences of altering 
                                  the timeline or interacting with different eras. Examples: H.G. Wells, Philip K. Dick, and Connie Willis.', 
                                  'Utopian/Dystopian: This style explores the implications of idealized or nightmarish societies, often serving 
                                  as commentary on current social issues. Examples: George Orwell, Aldous Huxley, and Margaret Atwood.']



                                  

                                  Question: {question}
                             """

In [27]:
prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

In [28]:
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

In [29]:
model = ChatOpenAI(model="gpt-4o-mini")
chain = (
            {"context": retriever | format_docs, "question": RunnablePassthrough()}
            | prompt
            | model
            | StrOutputParser()
         )

In [35]:
response = chain.invoke("""Using the themes, settings, styles and context provided, generate 10 diverse questions that are applicable 
                            across the document collection, taking into account the themes as guiding topics, settings as 
                            environments or backdrops for the inquiries, and styles to shape the tone or angle of each question. 
                            Aim for a mixture of questions about the motivations, interactions, ethical implications, societal 
                            impacts, and personal reflections that might emerge in documents with these characteristics. 
                            Attach the answers to the questions, with documents that support the answer.
                            The questions should not be too sophisticated
                        """)

In [36]:
print(response)

Here’s a set of 10 diverse questions along with their answers, drawing from the themes, settings, and styles of the context provided. Each question is designed to be relevant across the document collection, while the answers reference specific documents that support them.

### Questions and Answers

1. **Question:** What ethical dilemmas arise when humans interact with non-human intelligences in simulated environments?
   - **Answer:** The case of **Elysion**, where Dr. Mira Sorin grapples with the moral implications of creating a hyper-realistic virtual reality to interact with an artificial intelligence, raises questions about manipulation and the autonomy of the intelligence. (Document: *Elysion*)

2. **Question:** How do different settings, like urban centers or isolated research facilities, influence human perceptions of alien intelligence?
   - **Answer:** In **New Elysium**, the presence of The Synthetix in an urban center illustrates how the vibrant city life contrasts with the